## E06 设计不同的temperature对比输出的区别

In [3]:
words = open('names.txt', 'r').read().splitlines()

words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [4]:
import torch

N = torch.zeros((27*27, 27), dtype=torch.int32)  # 27*27 situation for taking two characters 

chars = ['.'] + sorted(set(''.join(words)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

In [5]:
xs, ys = [], []

for w in words:
    chs = ['.'] + ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1] * 27 + stoi[ch2]
        ix2 = stoi[ch3]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27*27, 27), generator=g, requires_grad=True)

number of examples:  228146


In [6]:
import torch.nn.functional as F


# gradient descent
for i in range(100):
    # forward pass
    # xenc = F.one_hot(xs, num_classes=27*27).float()
    # logits = xenc @ W

    #replace one-hot
    logits = W[xs]

    # # softmax
    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdims=True)
    # # loss
    # loss = -probs[torch.arange(num), ys].log().mean()
    
    # cross entropy loss
    loss = F.cross_entropy(logits, ys)

    # backward
    W.grad = None
    loss.backward()

    # update
    W.data += -100 * W.grad

loss.item()

2.3827080726623535

In [12]:
for temperature in [0.5, 0.8, 1.0, 1.5]:
    print(f'Temperature = {temperature}')

    g = torch.Generator().manual_seed(2147483647)

    for i in range(10):
        out = []
        ix1 = 0

        while True:
            logits = W[ix1]

            p = F.softmax(logits / temperature, dim=0)

            ix2 = torch.multinomial(
                p,
                num_samples=1,
                replacement=True,
                generator=g
            ).item()

            out.append(itos[ix2])

            if ix2 == 0:
                break

            ix1 = (ix1 % 27) * 27 + ix2

        print(''.join(out))

Temperature = 0.5
janiden.
ana.
ad.
aley.
aleigh.
kair.
la.
lee.
sa.
em.
Temperature = 0.8
juwjden.
ana.
ad.
pariah.
adin.
kair.
ltoni.
amarie.
kalinaariani.
evence.
Temperature = 1.0
juwjde.
jakasiz.
prefarie.
na.
kairritonian.
juwzvksiania.
yaniahviassdbyainrwibwlassjywnfyla.
te.
faveumtrifoetumj.
poytslena.
Temperature = 1.5
juwjdedianaqaz.
pxzfra.
adin.
koiirltohcassagezvkslaniauvdmileveackdbduilrwibwldsnjywnfylartbzmnvmumtryfoetumj.
pfnnslwnhosani.
coreayae.
ocfpypjabdmwebfmoifmwynvnwagaasnhmvfiusfszxhxdgosfmxtdnvic.
ir.
paupmvhmt.
rynrsra.


- T=0.5：分布更尖锐，模型倾向选择高概率字符，名字较短、较稳定，但多样性较低，例如 ana、aleigh、kair。
- T=0.8：质量与多样性比较均衡，整体效果最好，例如 pariah、amarie、evence。
- T=1.0：按照模型原始概率采样，多样性增加，但开始出现较长或不自然的结果。
- T=1.5：概率分布被压平，低概率字符更容易被选中，因此乱码和超长字符串明显增加。